# Macro

### Morans I
使用 Incremental Spatial Autocorrelation (ISA)

In [1]:
import numpy as np
import pandas as pd
from esda import Moran
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Polygon
from libpysal.weights import DistanceBand
from scipy.spatial import distance_matrix

dataA1 = pd.read_csv('../ComputedData/Accident/DataA1_with_MRT_Youbike_Parkinglot.csv')
dataA2 = pd.read_csv('../ComputedData/Accident/DataA2_with_MRT_Youbike_Parkinglot.csv')

In [2]:
filtered_A2 = dataA2[dataA2['當事者順位'] == 1]
# filtered_A2 = filtered_A2[filtered_A2['發生月份'] == 1]
print(filtered_A2.shape)

(317023, 54)


In [3]:
def create_hexagon(center_x, center_y, size):
    angles = np.linspace(0, 2 * np.pi, 7)
    return Polygon([
        (center_x + size * np.cos(angle), center_y + size * np.sin(angle))
        for angle in angles
    ])

def get_grid(data, hex_size):
    """
    # hexagon 大小 (degree)
    # 台灣約395,144 km
    hex_size = 1  # 度數，1 度 ≈ 111 公里
    """
    gdf = gpd.GeoDataFrame(data, geometry=gpd.points_from_xy(data['經度'], data['緯度']))

    gdf = gdf[
        (gdf['經度'] >= 119.7) & (gdf['經度'] <= 122.1) &
        (gdf['緯度'] >= 21.8) & (gdf['緯度'] <= 25.4)
    ]
    # 計算範圍 (bounding box)
    bounds = gdf.total_bounds  # (minx, miny, maxx, maxy)
    minx, miny, maxx, maxy = bounds

    # 六邊形的寬度和高度
    width = hex_size * 2
    height = np.sqrt(3) * hex_size

    # 計算橫向和縱向的hexagon間距
    x_spacing = width * 3/4
    y_spacing = height

    hexagons = []

    print('create hexagon')
    x = minx
    while x < maxx + width:
        y = miny
        while y < maxy + height:
            hex_center = (x, y)
            hexagon = create_hexagon(*hex_center, hex_size)
            hexagons.append(hexagon)
            y += y_spacing
        x += x_spacing

    # 每列交錯排列，蜂巢狀
    hexagons_shifted = []
    row = 0
    x = minx
    while x < maxx + width:
        y = miny + (y_spacing / 2 if row % 2 else 0) 
        while y < maxy + height:
            hex_center = (x, y)
            hexagon = create_hexagon(*hex_center, hex_size)
            hexagons_shifted.append(hexagon)
            y += y_spacing
        x += x_spacing
        row += 1

    print('get grid')
    hex_grid = gpd.GeoDataFrame(geometry=hexagons_shifted, crs='EPSG:4326')
    # 將事故點分配到 hexagon
    joined = gpd.sjoin(gdf, hex_grid, how='left', predicate='within')
    # 統計每個 hexagon 的事故數量
    hex_grid['num_accidents'] = joined.groupby('index_right').size()
    # 沒有事故的 hexagon 設為 0
    hex_grid['num_accidents'] = hex_grid['num_accidents'].fillna(0).astype(int)

    return hex_grid

def incremental_spatial_autocorrelation(gdf, value_col, min_dist=1000, max_dist=50000, step=1000):
    """
    gdf: CRS (meter)。
    value_col: 你要做 spatial autocorrelation 的欄位，如num_accidents
    min_dist: 最小距離 (公尺)
    max_dist: 最大距離 (公尺)
    step: 間隔 (公尺)
    """

    thresholds = np.arange(min_dist, max_dist + step, step)

    moran_I = []
    z_scores = []
    p_values = []

    centroids = gdf.centroid
    coords = np.array(list(zip(centroids.x, centroids.y)))
    y = gdf[value_col].values

    for thresh in thresholds:
        print(f"Processing threshold: {thresh} meters")
        w = DistanceBand(coords, threshold=thresh, binary=True, silence_warnings=True)
        w.transform = 'r'  # row-standardized
        moran = Moran(y, w)
        print(moran.z_norm)

        moran_I.append(moran.I)
        z_scores.append(moran.z_norm)  # ArcGIS 用的是 Z-score
        p_values.append(moran.p_norm)

    return thresholds, moran_I, z_scores, p_values


In [ ]:
hex_grid = get_grid(filtered_A2, hex_size=0.05)
hex_grid = hex_grid[hex_grid['num_accidents'] > 0] 

fig, ax = plt.subplots(figsize=(10, 10))

hex_grid.plot(
    column='num_accidents',  # 按事故數量填色
    cmap='OrRd',             # 紅色系列 (Orange-Red)
    legend=True,             # 顯示 colorbar
    edgecolor='black',       # 格子邊框顏色
    linewidth=0.2,           # 邊框細一點
    ax=ax
)

plt.title('Hex Grid of Accident Counts (num_accidents > 0)')
plt.axis('off')
plt.show()

In [ ]:
taiwan = gpd.read_file('../Data/OFiles_9e222fea-bafb-4436-9b17-10921abc6ef2/TOWN_MOI_1140318.shp')
taiwan = taiwan[(~taiwan['TOWNNAME'].isin(['旗津區', '頭城鎮'])) & 
                (~taiwan['COUNTYNAME'].isin(['金門縣', '連江縣', '澎湖縣']))]
taiwan = taiwan.to_crs(hex_grid.crs)  # 確保 CRS 一致

fig, ax = plt.subplots(figsize=(10, 10))
taiwan.plot(ax=ax, color='white', edgecolor='black')  # 底圖：台灣行政區
hex_grid.plot(
    column='num_accidents', 
    cmap='OrRd', 
    legend=True, 
    edgecolor='black', 
    linewidth=0.2, 
    alpha=0.6,
    ax=ax
)
plt.title('Hex Grid of Accident Counts (num_accidents > 0)')
plt.axis('off')
plt.show()

In [6]:
hex_grid = get_grid(filtered_A2, hex_size=0.05)
hex_grid = hex_grid[hex_grid['num_accidents'] > 0] 

# 投影成公尺
hex_grid = hex_grid.to_crs(epsg=3826)

# Incremental Spatial Autocorrelation
thresholds, moran_I, z_scores, p_values = incremental_spatial_autocorrelation(
    hex_grid, value_col='num_accidents', min_dist=5000, max_dist=30000, step=1000
)

create hexagon
get grid


/var/folders/w2/_g9w5yys0f171q4qqm469z1h0000gn/T/ipykernel_94360/4004869576.py:62: UserWarning: CRS mismatch between the CRS of left geometries and the CRS of right geometries.
Use `to_crs()` to reproject one of the input geometries to match the CRS of the other.

Left CRS: None
Right CRS: EPSG:4326

  joined = gpd.sjoin(gdf, hex_grid, how='left', predicate='within')


Processing threshold: 5000 meters
nan
Processing threshold: 6000 meters
nan
Processing threshold: 7000 meters
nan
Processing threshold: 8000 meters
nan
Processing threshold: 9000 meters


/Users/wangqiqian/opt/anaconda3/envs/ST-RTA/lib/python3.9/site-packages/esda/moran.py:219: RuntimeWarning: invalid value encountered in scalar divide
  self.VI_norm = v_num / v_den - (1.0 / (n - 1)) ** 2
/Users/wangqiqian/opt/anaconda3/envs/ST-RTA/lib/python3.9/site-packages/esda/moran.py:231: RuntimeWarning: invalid value encountered in scalar divide
  VIR = (A - B) / ((n - 1) * (n - 2) * (n - 3) * s02) - EI * EI
/Users/wangqiqian/opt/anaconda3/envs/ST-RTA/lib/python3.9/site-packages/esda/moran.py:238: RuntimeWarning: divide by zero encountered in scalar divide
  return self.n / self.w.s0 * inum / self.z2ss
/Users/wangqiqian/opt/anaconda3/envs/ST-RTA/lib/python3.9/site-packages/esda/moran.py:238: RuntimeWarning: invalid value encountered in scalar multiply
  return self.n / self.w.s0 * inum / self.z2ss


9.574311958123031
Processing threshold: 10000 meters
13.094907477929128
Processing threshold: 11000 meters
13.094907477929128
Processing threshold: 12000 meters
13.094907477929128
Processing threshold: 13000 meters
13.094907477929128
Processing threshold: 14000 meters
13.094907477929128
Processing threshold: 15000 meters
13.094907477929128
Processing threshold: 16000 meters
12.869825254676455
Processing threshold: 17000 meters
12.114048850826448
Processing threshold: 18000 meters
12.02616046779027
Processing threshold: 19000 meters
11.963322806150195
Processing threshold: 20000 meters
11.607660229356751
Processing threshold: 21000 meters
11.607660229356751
Processing threshold: 22000 meters
11.607660229356751
Processing threshold: 23000 meters
11.607660229356751
Processing threshold: 24000 meters
11.387650277055181
Processing threshold: 25000 meters
10.675733429995413
Processing threshold: 26000 meters
10.152094588540287
Processing threshold: 27000 meters
10.335463496082026
Processing 

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(thresholds / 1000, z_scores, marker='o')
plt.xlabel('Distance Threshold (km)')
plt.ylabel('Z-Score')
plt.title("Incremental Spatial Autocorrelation (ISA)")
plt.grid(True)
plt.show()

# 找 Z-score 最大的距離
best_idx = np.argmax(z_scores)
best_distance = thresholds[best_idx]
print(f"最佳分析距離 (m): {best_distance}")
print(f"Z-score: {z_scores[best_idx]:.4f}")
print(f"Moran's I: {moran_I[best_idx]:.4f}")

### GI

In [8]:
from libpysal.weights import DistanceBand
from esda import G_Local

best_distance = 10000

# 建立鄰接矩陣（以中心點）
centroids = hex_grid.centroid
coords = np.array(list(zip(centroids.x, centroids.y)))

w = DistanceBand(coords, threshold=best_distance, binary=True, silence_warnings=True)
w.transform = 'r'
y = hex_grid['num_accidents'].values
g_local = G_Local(y, w)

hex_grid['GiZScore'] = g_local.Zs  # 存到 hex_grid

print(hex_grid[['num_accidents', 'GiZScore']].head())


/Users/wangqiqian/opt/anaconda3/envs/ST-RTA/lib/python3.9/site-packages/esda/getisord.py:514: RuntimeWarning: invalid value encountered in divide
  z_scores = (statistic - expected_value) / np.sqrt(expected_variance)


    num_accidents  GiZScore
14              4 -0.278490
15              2 -0.310904
53             66  1.972469
54            285  0.927612
55            189 -0.130933


/Users/wangqiqian/opt/anaconda3/envs/ST-RTA/lib/python3.9/site-packages/esda/getisord.py:443: RuntimeWarning: invalid value encountered in divide
  self.z_sim = (self.Gs - self.EG_sim) / self.seG_sim


In [ ]:
taiwan = gpd.read_file('../Data/OFiles_9e222fea-bafb-4436-9b17-10921abc6ef2/TOWN_MOI_1140318.shp')
taiwan = taiwan[(~taiwan['TOWNNAME'].isin(['旗津區', '頭城鎮'])) & 
                (~taiwan['COUNTYNAME'].isin(['金門縣', '連江縣', '澎湖縣']))]
taiwan = taiwan.to_crs(hex_grid.crs)  # 確保 CRS 一致

hex_grid['hotspot'] = 'Not Significant'
hex_grid.loc[hex_grid['GiZScore'] > 1.65, 'hotspot'] = 'Hotspot'
hex_grid.loc[hex_grid['GiZScore'] < -1.65, 'hotspot'] = 'Coldspot'

fig, ax = plt.subplots(figsize=(10, 10))
taiwan.plot(ax=ax, color='white', edgecolor='black', linewidth=0.5)

hex_grid.plot(
    column='hotspot', 
    categorical=True, 
    cmap='coolwarm', 
    legend=True, 
    edgecolor='grey', 
    linewidth=0.2, 
    alpha=0.6,
    ax=ax
)

plt.title('Hotspot Analysis (Getis-Ord Gi*)')
plt.axis('off')
plt.show()